# Getting started with `neurofuzzy`

This notebook walks through loading a benchmark, training the neurofuzzy
semantic-similarity model, evaluating it, comparing against a baseline, and
visualizing predictions.

Companion code for Martinez-Gil et al. (2023), *Neurofuzzy semantic similarity
measurement*, Data & Knowledge Engineering 145, 102155.


## 1. Setup

Install the package from the repository root (`pip install -e ".[viz]"`).
All cells below run in a couple of minutes on a laptop.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neurofuzzy import NeurofuzzyModel
from neurofuzzy.data_loader import load_dataset, train_test_split
from neurofuzzy.baselines import MeanFeatureBaseline
from neurofuzzy.metrics import evaluate_all

import neurofuzzy
print('neurofuzzy', neurofuzzy.__version__)


## 2. Load a dataset

Each row is `ground_truth, feature_1, ..., feature_4`. We use the Miller-Charles
style benchmark bundled with the repository.


In [ ]:
X, y = load_dataset('../datasets/mc.txt', n_features=4)
print('X shape:', X.shape, '| y shape:', y.shape)
print('first pair  features:', X[0], '-> target', y[0])


## 3. Train/test split

We hold out 40% of the pairs to measure generalization.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_ratio=0.6, random_state=0)
print('train:', X_train.shape[0], 'pairs | test:', X_test.shape[0], 'pairs')


## 4. Fit the neurofuzzy model

The fuzzy controller has 40 bounded parameters optimized by Differential
Evolution. We use a small budget here for speed.


In [ ]:
model = NeurofuzzyModel(maxiter=40, popsize=8, train_ratio=1.0, random_state=0, verbose=False)
model.fit(X_train, y_train)
print('learned parameter vector length:', model.params_.shape)


## 5. Evaluate and compare to a baseline

We compare against the parameter-free *mean-of-features* baseline.


In [ ]:
nf_metrics = model.evaluate(X_test, y_test)
baseline = MeanFeatureBaseline().fit(X_train, y_train)
bl_metrics = evaluate_all(y_test, baseline.predict(X_test))

for name in ('pearson', 'spearman', 'mean_absolute_error'):
    print(f'{name:22s} neurofuzzy={nf_metrics[name]:.3f}  baseline={bl_metrics[name]:.3f}')


## 6. Visualize predictions

A scatter of predicted vs. true similarity on the test split. Points near the
diagonal are good predictions.


In [ ]:
pred = model.predict(X_test)
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='ideal')
plt.scatter(y_test, pred, s=50, label='neurofuzzy')
plt.xlabel('true similarity'); plt.ylabel('predicted similarity')
plt.title('MC test predictions'); plt.legend(); plt.tight_layout(); plt.show()


## 7. What to notice

Compare the model's **train** vs **test** correlation:



In [ ]:
print('train pearson:', round(model.evaluate(X_train, y_train)['pearson'], 3))
print('test  pearson:', round(nf_metrics['pearson'], 3))


On these small benchmarks the model fits the training data very well but
generalizes less well to held-out pairs — a classic overfitting signal. The
repository's benchmark harness quantifies this across many seeds with
confidence intervals and significance tests:

```bash
python -m neurofuzzy.benchmark --all --seeds 10
python -m neurofuzzy.visualize
```

**Exercises**

1. Re-run section 4 with `maxiter=120`. Does the train/test gap grow?
2. Swap in `BestSingleFeatureBaseline` and compare.
3. Try `datasets/geresid.txt` and explain the difference.
